# KAE GPU Runner (Kaggle)

Обёртка для RFC 0022 Runner. Вся серверная логика — в репозитории
(`src/agents/runner/`), блокнот только поднимает окружение и запускает её.
Поэтому изменения в раннере не требуют правки блокнота: он клонирует репозиторий
при каждом запуске.

**Порядок:** клонировать репу → поставить зависимости → секреты в env →
поднять cloudflared → запустить раннер.

## Секреты (Settings → Add-ons → Secrets)

| Секрет | Назначение |
|---|---|
| `KAE_MANAGER_URL` | куда слать `/runner/announce` |
| `KAE_RUNNER_TOKEN` | Bearer-токен Manager↔Runner |

При push через `bin/push-kaggle-runner.sh` значения подставляются в
плейсхолдеры на месте; при ручном запуске берутся из Kaggle Secrets.

## Два способа прислать страницу

`POST /infer` принимает либо картинку, либо ссылку:

```json
{"task": "vision", "image_b64": "…"}
{"task": "vision", "source_url": "https://…/book.pdf", "page": 3}
```

Вторая форма существует потому, что канал вызывающей стороны бывает много
уже канала раннера. На rpi5 замерено 1.7 КБ/с вверх против 4.6 МБ/с вниз:
заливка отрендеренной страницы стоит ~13 с при инференсе в 1–3 с. По ссылке
уходит несколько сотен байт, документ едет по каналу раннера и кэшируется —
книга скачивается один раз на все свои страницы.

Чтобы KAE начал слать ссылки, у агента в `agents.json` должно стоять
`"source_fetch": true`, а у документа — известный публичный URL. Иначе
поведение прежнее, с заливкой картинки.

In [ ]:
# 1. Репозиторий и зависимости.
!git clone --depth=1 https://github.com/4stm4/BookAssembler.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install fastapi 'uvicorn[standard]' transformers==4.49.0 qwen-vl-utils accelerate pymupdf
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
# 2. Секреты → env.
# bin/push-kaggle-runner.sh подставляет значения вместо плейсхолдеров на
# push-time, чтобы они не попадали в git. Если push не использовался
# (ручной запуск в UI), читаем Kaggle Secrets.
import os

_URL = '__KAE_MANAGER_URL__'
_TOKEN = '__KAE_RUNNER_TOKEN__'
if _URL.startswith('__') or _TOKEN.startswith('__'):
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    _URL = sec.get_secret('KAE_MANAGER_URL')
    _TOKEN = sec.get_secret('KAE_RUNNER_TOKEN')

os.environ['KAE_MANAGER_URL'] = _URL
os.environ['KAE_RUNNER_TOKEN'] = _TOKEN
os.environ.setdefault('KAE_RUNNER_LOADERS', 'qwen_vl')
os.environ.setdefault('KAE_RUNNER_IDLE_TIMEOUT', '900')
print('manager:', _URL)
print('loaders:', os.environ['KAE_RUNNER_LOADERS'])

In [ ]:
# 3. Кэш и качество рендера для запросов по ссылке.
# Раннеру не нужно экономить на картинке: он рендерит у себя, заливать её
# по сети никто не будет. DPI здесь — качество, которое видит модель.
os.environ.setdefault('KAE_RUNNER_SOURCE_CACHE', '/kaggle/working/sources')
os.environ.setdefault('KAE_RUNNER_RENDER_DPI', '150')
os.environ.setdefault('KAE_RUNNER_FETCH_TIMEOUT', '120')
print('cache :', os.environ['KAE_RUNNER_SOURCE_CACHE'])
print('dpi   :', os.environ['KAE_RUNNER_RENDER_DPI'])

In [ ]:
# 4. cloudflared: публичный URL для раннера.
import itertools
import re
import subprocess

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5005'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
for line in itertools.islice(proc.stdout, 300):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
assert url, 'cloudflared did not report a public URL'
os.environ['KAE_RUNNER_PUBLIC_URL'] = url
print('Runner will announce as:', url)

In [ ]:
# 5. Раннер (foreground). Сам прогреет warmup-задачи, объявится Manager'у
#    и завершится после KAE_RUNNER_IDLE_TIMEOUT секунд простоя.
import sys
sys.path.insert(0, '/kaggle/working/repo')
!KAE_RUNNER_HOST=0.0.0.0 python -m src.agents.runner